# job-funnel — Getting Started

Welcome! This notebook walks you through the **built** part of the pipeline (Stage 0 ingest → Stage 1 heuristic gate) hands-on, and shows where the **planned** stages plug in.

```
Sources ─► Normalize (JobPosting) ─► [Store 🚧] ─► Heuristic gate ─► [LLM score 🚧] ─► [Human review 🚧] ─► [Resume+CL 🚧] ─► [Apply 🚧]
```

Read [`docs/architecture.md`](../docs/architecture.md) alongside this for the full map.

**What you'll do**
1. Set up the environment
2. Meet the shared `JobPosting` contract
3. Fetch real jobs from Greenhouse and Ashby (with an offline fallback)
4. Clean descriptions and inspect them
5. Run the heuristic gate and read the pruning reasons
6. Write your own `FilterRule`
7. Sketch a new ingestor
8. Run the test suite

## 1. Setup

From the repo root:

```bash
uv sync            # installs runtime + dev deps (pytest, jupyterlab, ruff)
uv run jupyter lab # open this notebook with the project's .venv kernel
```

The cell below makes sure we're running from the repo root so relative paths work.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
REPO = Path.cwd()
print("repo root:", REPO)
assert (REPO / "pyproject.toml").exists(), "start the notebook from inside the job-funnel repo"

## 2. The shared contract: `JobPosting`

Every source converts its own JSON into this one pydantic model. Downstream stages never see ATS-specific fields.

- Identity is `(source, external_id)`.
- `remote` is the source's structured flag when it has one (`None` = unknown); `is_remote` falls back to looking for "remote" in `location`.
- Pydantic validates on construction, so a posting with a bad/missing `url` raises and the ingestor drops it.

In [ ]:
from pydantic import ValidationError
from job_funnel.schemas import JobPosting

job = JobPosting(
    source="demo", external_id="42", title="Senior Data Scientist", company="ACME",
    description="Build models.", location="Remote - US", url="https://example.com/jobs/42",
)
print(job.model_dump_json(indent=2))
print("is_remote:", job.is_remote)

try:
    JobPosting(**{**job.model_dump(), "url": "not-a-url"})
except ValidationError as e:
    print("\nvalidation error (expected):", e.errors()[0]["msg"])

## 3. Fetch jobs from two sources

Both ingestors subclass `BaseIngestor`, which does the HTTP GET and hands the JSON to the subclass's `_parse`.

| Source | Endpoint | Board token example |
|---|---|---|
| Greenhouse | `boards-api.greenhouse.io/v1/boards/{token}/jobs?content=true` | `discord` |
| Ashby | `api.ashbyhq.com/posting-api/job-board/{token}` | `openai` |

The board token is the slug in a company's careers URL (e.g. `jobs.ashbyhq.com/openai`).

`ingest_all` fetches all sources **concurrently** and skips (but logs) any that fail. If you're offline we fall back to the test fixtures by injecting an `httpx.MockTransport` client — the same trick the unit tests use. Notebooks support top-level `await`, so we can call the async code directly.

In [ ]:
import json
import httpx

from job_funnel.ashby_ingestor import AshbyIngestor
from job_funnel.greenhouse_ingestor import GreenhouseIngestor
from job_funnel.pipeline import ingest_all

OFFLINE = False  # set True to skip the network entirely


def fixture_client(name: str) -> httpx.AsyncClient:
    payload = json.loads((REPO / "tests/fixtures" / name).read_text())
    return httpx.AsyncClient(transport=httpx.MockTransport(lambda req: httpx.Response(200, json=payload)))


async def fetch(offline: bool):
    if offline:
        return await ingest_all([
            GreenhouseIngestor("acme", client=fixture_client("greenhouse_jobs.json")),
            AshbyIngestor("acme", client=fixture_client("ashby_jobs.json")),
        ])
    return await ingest_all([GreenhouseIngestor("discord"), AshbyIngestor("openai")])


jobs = await fetch(OFFLINE)
if not jobs and not OFFLINE:
    print("\nNo live jobs (offline?) -> using fixtures")
    jobs = await fetch(True)

from collections import Counter
print("\njobs per source:", dict(Counter(j.source for j in jobs)))

## 4. Descriptions: why `clean_html` matters

Greenhouse returns description HTML whose tags are **entity-escaped** (`&lt;p&gt;` instead of `<p>`). If you strip tags *before* unescaping, the tags survive and pollute the text we later send to the LLM. `job_funnel.text.clean_html` unescapes first, strips `<script>/<style>` and tags, then collapses whitespace.

In [ ]:
from job_funnel.text import clean_html

raw = "&lt;div&gt;&lt;p&gt;Build &lt;strong&gt;ML&lt;/strong&gt; models &amp;amp; ship them.&lt;/p&gt;&lt;/div&gt;"
print("raw:    ", raw)
print("cleaned:", clean_html(raw))

sample = jobs[0]
print(f"\n{sample.source} | {sample.title} | {sample.location}")
print(sample.description[:400], "...")
import re
TAG = re.compile(r"</?[a-zA-Z][^>]*>|&lt;/?[a-zA-Z]")  # real tags, not text like "<18 users"
dirty = [j for j in jobs if TAG.search(j.description)]
print(f"\njobs with leftover HTML tags: {len(dirty)}")

## 5. Stage 1 — the heuristic gate

Rules run in order and the gate **short-circuits** on the first failure, returning that rule's reason:

1. `AntiKeywordRule` — drop titles like sales / recruiter / frontend
2. `TitleTaxonomyRule` — keep DS / ML / AI / LLM / quant titles
3. `LocationRule(require_remote=...)` — optionally require remote

The goal is **high recall**: it's fine to let borderline jobs through, the LLM stage will judge them. What we must avoid is silently dropping good jobs, which is why every rejection carries a reason.

In [ ]:
from job_funnel.pipeline import apply_gate, default_gate

surviving, pruned = apply_gate(default_gate(), jobs)
print(f"ingested={len(jobs)}  surviving={len(surviving)}  prune rate={len(pruned) / len(jobs):.1%}\n")

print("Top rejection reasons (by rule):")
print(Counter(reason.split("'")[0] + "..." + reason.split("'")[-1] for _, reason in pruned).most_common())

print("\nSample survivors:")
for j in surviving[:10]:
    print(f"  [{j.source:<14}] {j.title:<60} remote={j.is_remote}")

**Exercise:** look through `pruned` for false negatives — titles you *would* want that got dropped (e.g. "Applied Scientist", "Research Engineer"). Those are signals to widen `TitleTaxonomyRule`.

In [ ]:
for j, reason in pruned[:15]:
    print(f"  {j.title:<60} -> {reason}")

## 6. Write your own rule

A rule is any `FilterRule` whose `evaluate(job)` returns `(True, None)` or `(False, "reason")`. Here's one that rejects senior-leadership titles, inserted into a custom gate:

In [ ]:
import re
from job_funnel.heuristic_gate import AntiKeywordRule, FilterRule, HeuristicGate, LocationRule, TitleTaxonomyRule


class SeniorityRule(FilterRule):
    def __init__(self):
        self.pattern = re.compile(r"\b(director|vp|head of|chief)\b", re.IGNORECASE)

    def evaluate(self, job):
        if self.pattern.search(job.title):
            return False, f"Title '{job.title}' is too senior."
        return True, None


my_gate = HeuristicGate(rules=[AntiKeywordRule(), TitleTaxonomyRule(), SeniorityRule(), LocationRule(require_remote=True)])
my_surviving, my_pruned = apply_gate(my_gate, jobs)
print(f"default gate: {len(surviving)} survivors | custom gate (remote-only, no execs): {len(my_surviving)}")
for j in my_surviving[:10]:
    print("  ", j.title, "|", j.location)

## 7. Adding a new source

Subclass `BaseIngestor`: set `source`, implement `api_url` and `_parse`. Then add a JSON fixture + test under `tests/` and register it in `pipeline.main`. A sketch for Lever (not part of the package yet):

```python
from job_funnel.base_ingestor import BaseIngestor
from job_funnel.schemas import JobPosting

class LeverIngestor(BaseIngestor):
    source = "lever_api"

    @property
    def api_url(self) -> str:
        return f"https://api.lever.co/v0/postings/{self.company_token}?mode=json"

    def _parse(self, payload):
        ...  # note: Lever returns a bare JSON list, not {"jobs": [...]}, so also adapt fetch/count
```

Tip: inspect a real payload first (`httpx.get(url).json()`) and check which fields are reliable — e.g. Ashby sets `isRemote=True` on many *Hybrid* roles, so we trust `workplaceType` first.

## 8. Run the tests

All tests are offline (mocked HTTP), so they run in well under a second:

```bash
uv run pytest -q
uv run ruff check src tests
```

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no"], capture_output=True, text=True, cwd=REPO)
print(result.stdout[-500:])

## 9. What's next (planned stages)

| Stage | Idea | Where it plugs in |
|---|---|---|
| Store 🚧 | SQLite, dedup on `(source, external_id)`, status column | between ingest and gate |
| LLM score 🚧 | local vLLM, JD × candidate profile → JSON `{score, gaps, rationale}` | on `surviving` |
| Human review 🚧 | approve / reject with reason; feeds back into rules & prompts | on scored jobs |
| Resume + CL 🚧 | tailor master resume + cover letter per approved job | on approved jobs |
| Apply 🚧 | manual now, automated later — always behind human approval | last |

See `docs/architecture.md` for the lifecycle state machine and open design questions.